In [1]:
import pandas as pd
from sklearn.utils import resample
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [2]:
train = pd.read_csv(
    "D:/Prithiv/AIML PROJECT/DATASET/train.csv"
)
dev = pd.read_csv(
    "D:/Prithiv/AIML PROJECT/DATASET/dev.csv"
)
test = pd.read_csv(
    "D:/Prithiv/AIML PROJECT/DATASET/test.csv"
)

print("Train:", train.shape)
print("Dev:", dev.shape)
print("Test:", test.shape)

Train: (5512, 3)
Dev: (787, 3)
Test: (1576, 2)


In [3]:
X_train = train["text"].astype(str)
y_train = train["label"]
X_dev = dev["text"].astype(str)
y_dev = dev["label"]
X_test = test["text"].astype(str)

In [4]:
print("Original training distribution:")
print(y_train.value_counts())

print("\nPercentage:")
print(y_train.value_counts(normalize=True) * 100)

Original training distribution:
label
0    3415
1    2097
Name: count, dtype: int64

Percentage:
label
0    61.955733
1    38.044267
Name: proportion, dtype: float64


In [5]:
train_temp = train[["text", "label"]].copy()

majority = train_temp[train_temp["label"] == 0]
minority = train_temp[train_temp["label"] == 1]

print("Majority:", len(majority))
print("Minority:", len(minority))

Majority: 3415
Minority: 2097


In [6]:
minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

In [7]:
train_balanced = pd.concat([
    majority,
    minority_oversampled
])

train_balanced = train_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(train_balanced["label"].value_counts())

label
1    3415
0    3415
Name: count, dtype: int64


In [8]:
X_train = train_balanced["text"].astype(str)
y_train = train_balanced["label"]

X_dev = dev["text"].astype(str)
y_dev = dev["label"]
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_features=20000
)

X_train_bow = vectorizer.fit_transform(X_train)
X_dev_bow = vectorizer.transform(X_dev)

print("Train BoW:", X_train_bow.shape)
print("Dev BoW:", X_dev_bow.shape)
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train_bow,
    y_train
)
y_pred = model.predict(X_dev_bow)

Train BoW: (6830, 20000)
Dev BoW: (787, 20000)


In [9]:
print(classification_report(y_dev, y_pred))

print("Accuracy:",
      accuracy_score(y_dev, y_pred))

print("Macro Precision:",
      precision_score(y_dev, y_pred, average="macro"))

print("Macro Recall:",
      recall_score(y_dev, y_pred, average="macro"))

print("Macro F1:",
      f1_score(y_dev, y_pred, average="macro"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_dev, y_pred))

              precision    recall  f1-score   support

           0       0.76      0.90      0.82       485
           1       0.77      0.55      0.64       302

    accuracy                           0.76       787
   macro avg       0.77      0.72      0.73       787
weighted avg       0.76      0.76      0.75       787

Accuracy: 0.7636594663278272
Macro Precision: 0.7659677708730896
Macro Recall: 0.7226633440294941
Macro F1: 0.7318657405372138

Confusion Matrix:
[[436  49]
 [137 165]]


In [12]:
from sklearn.feature_extraction.text import  TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000
)

X_train_tfidf = tfidf_vectorizer.fit_transform(
    train_balanced["text"].astype(str)
)

X_dev_tfidf = tfidf_vectorizer.transform(
    dev["text"].astype(str)
)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Dev TF-IDF shape:", X_dev_tfidf.shape)

Train TF-IDF shape: (6830, 20000)
Dev TF-IDF shape: (787, 20000)


In [13]:
y_train = train_balanced["label"]
y_dev = dev["label"]

tfidf_model = LogisticRegression(max_iter=1000)

tfidf_model.fit(X_train_tfidf, y_train)

y_pred_tfidf = tfidf_model.predict(X_dev_tfidf)

tfidf_accuracy = accuracy_score(y_dev, y_pred_tfidf)
tfidf_precision = precision_score(y_dev, y_pred_tfidf, average="macro")
tfidf_recall = recall_score(y_dev, y_pred_tfidf, average="macro")
tfidf_f1 = f1_score(y_dev, y_pred_tfidf, average="macro")

print("Oversampling + TF-IDF + Logistic Regression")
print("---------------------------------------------")
print("Accuracy       :", tfidf_accuracy)
print("Macro Precision:", tfidf_precision)
print("Macro Recall   :", tfidf_recall)
print("Macro F1       :", tfidf_f1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_dev, y_pred_tfidf))

Oversampling + TF-IDF + Logistic Regression
---------------------------------------------
Accuracy       : 0.6950444726810674
Macro Precision: 0.6758952742467044
Macro Recall   : 0.6632450331125828
Macro F1       : 0.6668360003386673

Confusion Matrix:
[[388  97]
 [143 159]]
